# Adaptive RAG: Route by Evidence Need, Not Prompt Length

| Field | Value |
|---|---|
| Stage | Corrective and adaptive RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Adaptive RAG selects the cheapest branch that can satisfy the evidence requirement, with a shared step and cost budget.

## 30-Second Summary

A transparent router sends a direct fact lookup to one retrieval, a comparison to two retrievals plus synthesis, and an unsupported weather request to abstention.

## Why This Matters

Running the most capable workflow for every query wastes latency and cost; running the simplest one for every query misses compound evidence needs.

## Scope

| Covers | Does not cover |
|---|---|
| Route labels, branch contracts, coverage, step/cost budget, abstention | Learned router, online model pricing, dynamic tool marketplace |


## Mental Model

```text
classify evidence need -> direct | compare | abstain -> enforce budget -> cited result
```


In [1]:
POLICIES = {"Atlas": {"days": 30, "source": "atlas"}, "Beacon": {"days": 90, "source": "beacon"}}
cases = [
    ("How long does Atlas retain logs?", "direct"),
    ("Which retains logs longer, Atlas or Beacon?", "compare"),
    ("Will it rain tomorrow?", "abstain"),
]
MAX_STEPS = 3
MAX_COST = 3


## How It Works

The route is an evidence-shape label. Each branch declares expected steps and cost units; a controller rejects branches that exceed the shared budget before execution.


## Baseline

Always using the comparison branch retrieves two sources even for a one-fact question and still cannot answer weather.


In [2]:
baseline_cost = [3, 3, 3]
baseline_total_cost = sum(baseline_cost)
baseline_total_cost


9

## Technique Implementation

The router distinguishes a single known entity, a cross-entity comparison, and unsupported scope.


In [3]:
def route(query: str) -> str:
    lower = query.lower()
    if "weather" in lower or "rain" in lower: return "abstain"
    if "which" in lower or "compare" in lower: return "compare"
    return "direct"

def run(query: str) -> dict:
    branch = route(query)
    if branch == "abstain": return {"branch": branch, "steps": 1, "cost": 0, "sources": []}
    if branch == "direct": return {"branch": branch, "steps": 2, "cost": 1, "sources": [POLICIES["Atlas"]["source"]], "answer": "30 days"}
    longer = max(POLICIES, key=lambda name: POLICIES[name]["days"])
    return {"branch": branch, "steps": 3, "cost": 3, "sources": ["atlas", "beacon"], "answer": f"{longer}: {POLICIES[longer]['days']} days"}

results = [run(query) for query, _ in cases]
results


[{'branch': 'direct',
  'steps': 2,
  'cost': 1,
  'sources': ['atlas'],
  'answer': '30 days'},
 {'branch': 'compare',
  'steps': 3,
  'cost': 3,
  'sources': ['atlas', 'beacon'],
  'answer': 'Beacon: 90 days'},
 {'branch': 'abstain', 'steps': 1, 'cost': 0, 'sources': []}]

## Controlled Experiment

We score route accuracy, assert every branch stays within budget, and compare total cost with the always-complex baseline.


In [4]:
route_accuracy = sum(result["branch"] == expected for result, (_, expected) in zip(results, cases)) / len(cases)
adaptive_cost = sum(result["cost"] for result in results)
within_budget = all(result["steps"] <= MAX_STEPS and result["cost"] <= MAX_COST for result in results)
experiment = {"route_accuracy": route_accuracy, "adaptive_cost": adaptive_cost, "baseline_cost": baseline_total_cost, "within_budget": within_budget}
experiment


{'route_accuracy': 1.0,
 'adaptive_cost': 4,
 'baseline_cost': 9,
 'within_budget': True}

## Evaluation

The router labels **3/3** cases correctly and uses **4 cost units instead of 9**, while every branch stays within three steps. Savings depend on the chosen cost model and case mix.


In [5]:
assert experiment == {"route_accuracy": 1.0, "adaptive_cost": 4, "baseline_cost": 9, "within_budget": True}
assert results[1]["sources"] == ["atlas", "beacon"] and results[2]["branch"] == "abstain"
print("Adaptive-RAG checks passed.")


Adaptive-RAG checks passed.


## Decision Guide

| Evidence need | Branch |
|---|---|
| One source/fact | Direct retrieval |
| Multiple sources/derived result | Compound workflow |
| Weak retrieval | Corrective branch |
| Unsupported/out of scope | Abstain |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Expensive simple queries | Over-routing | Complexity labels/cost objective |
| Missing comparison clause | Under-routing | Evidence-coverage tests |
| Branch loop | No shared budget | Global step ceiling |
| Silent route drift | No labels | Route confusion matrix |


## Production Notes

### Observability
Track route label/confidence, branch, steps, evidence coverage, latency, cost, and terminal reason.

### Safety and Guardrails
All branches enforce the same ACL and citation policy.

### Latency and Cost
Optimize expected quality under explicit p95 and spend budgets.


## Practice

Add a weak-evidence case that routes to corrective RAG and remains within the global budget.

## Recall

Toggle - Recall: What should routing predict?
The evidence/workflow need, not merely query length.

Toggle - Recall: What bounds all branches?
Shared quality, safety, step, latency, and cost contracts.

## Sources

- [Adaptive-RAG](https://arxiv.org/abs/2403.14403)
- Repository-owned synthetic routing fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the labeled routing fixture | Add a corrective branch and calibration curves |
